# Demo MCTS + PPO

# Estado actual del proyecto

Resumen actual:

- **Formato objetivo:** `gen9vgc2026regi`.
- **Equipo usado:** `team.txt`.
- **Modelo actual:** `MCTS + PPO`, cargado desde el ultimo checkpoint disponible segun la lista de `DEFAULT_CHECKPOINTS`.
- **Entrenamiento mas reciente:** `alphazero_mcts_ppo_offline_d2`, planificado para 60 iteraciones; el log disponible llega hasta la iteracion 10 guardada y deja registrada la iteracion 11 como iniciada.
- **Datos de entrenamiento:** rollouts offline en `data/alphazero/rollouts_offline_d2.jsonl`, con ventana de entrenamiento de 20.000 muestras.
- **Servicios necesarios:** `showdown` para jugar partidas, `showdown-sim` para simular ramas, y bridge de estado vivo en `http://showdown:9002`.
- **Politicas jugables en `play.py`:** `random`, `greedy` y `alphazero_mcts`.

Estado de calidad observado hasta ahora:

- Contra `random`, el modelo suele ganar con bastante frecuencia, aunque depende de la muestra.
- Contra `greedy`, los resultados recientes se ven cercanos a 50/50; hay corridas con peor desempeno y otra con empate 15/30.
- El modelo todavia esta limitado por busqueda depth 2.

Modelos/checkpoints ya disponibles en el repo pero no todos estan comparados automaticamente en esta notebook:

- `alphazero_pretrain_robust_v2`: checkpoint de preentrenamiento robusto usado como punto inicial.
- `alphazero_mcts_ppo_d2_required_v6`, `d2_required_v4`, `d2_robust_v2`, `d2`, `d1`, `d4`: variantes online/diagnosticas con distintos ajustes de profundidad, simulador y robustez.
- `alphazero_mcts_ppo_offline_d2`: modelo offline actual de referencia para la demo.
- `alphazero_bc` y checkpoints `smoke`: utiles como pruebas o baselines tecnicos, pero no necesariamente como modelos finales.

Pendientes razonables para incluir despues:

- Comparar automaticamente varios checkpoints entre si, no solo el checkpoint principal.
- Incluir una celda de evaluacion offline rapida para comparar modelos sin jugar partidas completas por websocket.
- Separar en la demo un baseline `pretrain-only` frente al modelo ya ajustado con MCTS+PPO.
- Continuar las 50 iteraciones restantes del entrenamiento actual y contrastar metricas antes/despues.
- Revisar por que un entrenamiento anterior se detuvo alrededor de la iteracion 23.
- Incorporar o comparar los modelos pendientes:
  - **AlphaZero-style MCTS + PPO:** modelo actual de referencia. Combina busqueda Monte Carlo con una red de politica+valor y entrenamiento PPO.
  - **Counterfactual Regret Minimization (CFR):** alternativa pensada para simultaneidad e informacion incompleta, inspirada en poker.
  - **Transformer sobre el estado del juego:** reemplazar el MLP plano por tokens por Pokemon y atencion entre aliados/rivales.
  - **Curriculum Learning estructurado:** pasar automaticamente de random a greedy, reglas basicas, self-play y humanos segun win rate objetivo.
  - **DreamerV3 / world model:** aprender un modelo latente del entorno para entrenar con simulaciones imaginadas y reducir costo de partidas reales.


## 1. Configuracion

In [23]:
from __future__ import annotations

import json
import os
import re
import subprocess
import textwrap
import time
from http.client import RemoteDisconnected
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "docker-compose.yml").exists() and (candidate / "play.py").exists():
            return candidate
    raise RuntimeError("No pude encontrar la raiz del repo. Ejecuta la notebook dentro del proyecto Labo3.")


ROOT = find_repo_root(Path.cwd()).resolve()
os.chdir(ROOT)

DEFAULT_CHECKPOINTS = [
    Path("checkpoints/alphazero_mcts_ppo_offline_d2/best.pt"),
    Path("checkpoints/alphazero_mcts_ppo_d2_required_v6/best.pt"),
    Path("checkpoints/alphazero_mcts_ppo_d2_required_v4/best.pt"),
    Path("checkpoints/alphazero_mcts_ppo_d2_robust_v2/best.pt"),
    Path("checkpoints/alphazero_mcts_ppo_d2/best.pt"),
]
CHECKPOINT = next((path for path in DEFAULT_CHECKPOINTS if path.exists()), DEFAULT_CHECKPOINTS[0])
FORMAT = "gen9vgc2026regi"
TEAM = "team.txt"
SERVER = "showdown:8000"
LIVE_STATE_URL = "http://showdown:9002"
SIMULATOR_URL = "http://showdown-sim:9001"
OUTPUT_DIR = Path("notebooks/demo_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REBUILD_SERVICES = True

# Rivales disponibles en play.py. Si self-play tarda demasiado, quitar "alphazero_mcts".
RIVALS = ["random", "greedy", "alphazero_mcts"]

if not CHECKPOINT.exists():
    available = sorted(Path("checkpoints").glob("*/best.pt"), key=lambda path: path.stat().st_mtime, reverse=True)
    print("No encontre el checkpoint default. Checkpoints disponibles:")
    for path in available[:20]:
        print("-", path)
    raise FileNotFoundError(f"No existe el checkpoint: {CHECKPOINT}. Cambia CHECKPOINT por uno de la lista.")
assert Path(TEAM).exists(), f"No existe el equipo: {TEAM}"
print(f"Repo: {ROOT}")
print(f"Checkpoint: {CHECKPOINT}")

Repo: C:\Users\rodri\Desktop\R\UCA\2026 1ro\lab3\Labo3
Checkpoint: checkpoints\alphazero_mcts_ppo_offline_d2\best.pt


## 2. Helpers

In [24]:
def run_cmd(args: list[str], *, timeout: int | None = None, save_to: Path | None = None) -> str:
    print("$ " + " ".join(args))
    proc = subprocess.Popen(
        args,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    lines: list[str] = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    code = proc.wait(timeout=timeout)
    output = "".join(lines)
    if save_to is not None:
        save_to.parent.mkdir(parents=True, exist_ok=True)
        save_to.write_text(output, encoding="utf-8")
    if code != 0:
        raise RuntimeError(f"Comando fallo con exit code {code}")
    return output


def docker_compose(*args: str, timeout: int | None = None, save_to: Path | None = None) -> str:
    return run_cmd(["docker", "compose", *args], timeout=timeout, save_to=save_to)


def wait_http(url: str, *, attempts: int = 30, delay: float = 2.0) -> str:
    last_error: Exception | None = None
    for attempt in range(1, attempts + 1):
        try:
            with urlopen(url, timeout=10) as response:
                body = response.read().decode("utf-8", errors="replace")
                print(f"{url} -> HTTP {response.status}: {body}")
                return body
        except (OSError, URLError, RemoteDisconnected) as exc:
            last_error = exc
            print(f"Esperando {url} ({attempt}/{attempts}): {type(exc).__name__}: {exc}")
            time.sleep(delay)
    raise RuntimeError(f"{url} no respondio correctamente. Ultimo error: {last_error}")


def show_service_diagnostics() -> None:
    print("\n--- docker compose ps ---")
    try:
        docker_compose("ps")
    except Exception as exc:
        print(f"No pude consultar ps: {exc}")
    for service in ["showdown", "showdown-sim"]:
        print(f"\n--- logs {service} ---")
        try:
            docker_compose("logs", "--tail", "80", service)
        except Exception as exc:
            print(f"No pude leer logs de {service}: {exc}")


def play_command(rival: str, n: int, *, simulations: int = 128, depth: int = 2) -> list[str]:
    cmd = [
        "run", "--rm", "trainer", "python", "-u", "play.py",
        "--mode", "challenge",
        "--n", str(n),
        "--p1", "alphazero_mcts",
        "--p2", rival,
        "--server", SERVER,
        "--format", FORMAT,
        "--team", TEAM,
        "--alphazero-checkpoint", str(CHECKPOINT).replace("\\", "/"),
        "--alphazero-simulations", str(simulations),
        "--alphazero-depth", str(depth),
        "--alphazero-max-candidates", "96",
        "--alphazero-simulator-max-choices", "8",
        "--alphazero-simulator-opponent-policy", "robust",
        "--alphazero-simulator-robust-worst-weight", "0.35",
        "--alphazero-simulator-timeout", "180",
        "--alphazero-live-state-url", LIVE_STATE_URL,
        "--alphazero-require-simulator",
        "--alphazero-device", "cpu",
        "--battle-timeout", "1800",
    ]
    return cmd


def parse_play_output(output: str, rival: str, n: int) -> dict:
    principal = re.search(r"Principal\s+(\S+)\s+victorias:\s+(\d+)\s*/\s*(\d+)", output)
    opponent = re.search(r"Rival\s+(\S+)\s+victorias:\s+(\d+)\s*/\s*(\d+)", output)
    games = re.findall(r"\[(VICTORIA|DERROTA)\]\s+(battle-[^\s]+)\s+-\s+(\d+)\s+turnos", output)
    final_lines = re.findall(r"Final:\s+(battle-[^\s]+)\s+->\s+(VICTORIA|DERROTA)\s+en\s+(\d+)\s+turnos", output)
    records = games or [(result, battle, turns) for battle, result, turns in final_lines]
    turns = [int(turns) for _, _, turns in records]
    wins = int(principal.group(2)) if principal else sum(1 for result, _, _ in records if result == "VICTORIA")
    total = int(principal.group(3)) if principal else (len(records) or n)
    rival_wins = int(opponent.group(2)) if opponent else total - wins
    return {
        "rival": rival,
        "n": total,
        "wins": wins,
        "losses": rival_wins,
        "win_rate": wins / total if total else 0.0,
        "avg_turns": sum(turns) / len(turns) if turns else None,
        "min_turns": min(turns) if turns else None,
        "max_turns": max(turns) if turns else None,
        "parsed_games": len(records),
        "raw_output_chars": len(output),
    }


def show_table(rows: list[dict]) -> None:
    if not rows:
        print("Sin datos para mostrar.")
        return
    columns = sorted({key for row in rows for key in row})
    widths = {col: max(len(col), *(len(str(row.get(col, ""))) for row in rows)) for col in columns}
    print(" | ".join(col.ljust(widths[col]) for col in columns))
    print("-+-".join("-" * widths[col] for col in columns))
    for row in rows:
        print(" | ".join(str(row.get(col, "")).ljust(widths[col]) for col in columns))

## 3. Levantar y verificar servicios

In [25]:
compose_up_args = ["up", "-d"]
if REBUILD_SERVICES:
    compose_up_args += ["--build", "--force-recreate"]
compose_up_args += ["showdown", "showdown-sim"]

docker_compose(*compose_up_args)
docker_compose("ps")

try:
    wait_http("http://localhost:9002/health")
    wait_http("http://localhost:9001/health")
except Exception:
    show_service_diagnostics()
    raise


$ docker compose up -d --build --force-recreate showdown showdown-sim
#1 [internal] load local bake definitions
#1 reading from stdin 1.09kB done
#1 DONE 0.0s

#2 [showdown-sim internal] load build definition from Dockerfile.showdown
#2 transferring dockerfile: 897B done
#2 DONE 0.0s

#3 [showdown internal] load metadata for docker.io/library/node:18-slim
#3 DONE 1.2s

#4 [showdown-sim internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [showdown-sim 1/7] FROM docker.io/library/node:18-slim@sha256:f9ab18e354e6855ae56ef2b290dd225c1e51a564f87584b9bd21dd651838830e
#5 resolve docker.io/library/node:18-slim@sha256:f9ab18e354e6855ae56ef2b290dd225c1e51a564f87584b9bd21dd651838830e 0.0s done
#5 DONE 0.0s

#6 [showdown internal] load build context
#6 transferring context: 136B done
#6 DONE 0.0s

#7 [showdown 3/7] WORKDIR /showdown
#7 CACHED

#8 [showdown 6/7] COPY tools/patch_showdown_live_state.js /tmp/patch_showdown_live_state.js
#8 CACHED

#9 [showdown 5/7] COPY too

## 4. Inspeccionar el checkpoint

In [26]:
checkpoint_path_for_container = str(CHECKPOINT).replace("\\", "/")
inspect_code = "\n".join([
    "import json",
    "import torch",
    f"path = {json.dumps(checkpoint_path_for_container)}",
    "payload = torch.load(path, map_location='cpu')",
    "state = payload.get('model_state_dict', {})",
    "params = sum(t.numel() for t in state.values() if hasattr(t, 'numel'))",
    "summary = {",
    "    'path': path,",
    "    'keys': sorted(payload.keys()),",
    "    'model_config': payload.get('model_config'),",
    "    'trained_rows': payload.get('trained_rows'),",
    "    'parameter_count': params,",
    "}",
    "print(json.dumps(summary, indent=2))",
])
checkpoint_info = docker_compose("run", "--rm", "trainer", "python", "-c", inspect_code, save_to=OUTPUT_DIR / "checkpoint_info.txt")

$ docker compose run --rm trainer python -c import json
import torch
path = "checkpoints/alphazero_mcts_ppo_offline_d2/best.pt"
payload = torch.load(path, map_location='cpu')
state = payload.get('model_state_dict', {})
params = sum(t.numel() for t in state.values() if hasattr(t, 'numel'))
summary = {
    'path': path,
    'keys': sorted(payload.keys()),
    'model_config': payload.get('model_config'),
    'trained_rows': payload.get('trained_rows'),
    'parameter_count': params,
}
print(json.dumps(summary, indent=2))
time="2026-05-04T21:01:05-03:00" level=warning msg="Found orphan containers ([alphazero-train-offline-v2 alphazero-train-offline]) for this project. If you removed or renamed this service in your compose file, you can run this command with the --remove-orphans flag to clean it up."
 Container showdown-sim  Running
 Container showdown  Running
{
  "path": "checkpoints/alphazero_mcts_ppo_offline_d2/best.pt",
  "keys": [
    "model_config",
    "model_state_dict",
    "train

## 5. Una partida contra cada rival

Corre una partida por rival disponible y parsea metricas basicas. Esta celda sirve para mostrar la toma de decisiones completa turno a turno.

In [ ]:
single_game_metrics = []
single_game_outputs = {}

for rival in RIVALS:
    print("\n" + "=" * 80)
    print(f"AlphaZero MCTS vs {rival} - 1 partida")
    output_path = OUTPUT_DIR / f"single_vs_{rival}.txt"
    output = docker_compose(*play_command(rival, 1), save_to=output_path)
    single_game_outputs[rival] = output
    single_game_metrics.append(parse_play_output(output, rival, 1))

show_table(single_game_metrics)


AlphaZero MCTS vs random - 1 partida
$ docker compose run --rm trainer python -u play.py --mode challenge --n 1 --p1 alphazero_mcts --p2 random --server showdown:8000 --format gen9vgc2026regi --team team.txt --alphazero-checkpoint checkpoints/alphazero_mcts_ppo_offline_d2/best.pt --alphazero-simulations 128 --alphazero-depth 2 --alphazero-max-candidates 96 --alphazero-simulator-max-choices 8 --alphazero-simulator-opponent-policy robust --alphazero-simulator-robust-worst-weight 0.35 --alphazero-simulator-timeout 180 --alphazero-live-state-url http://showdown:9002 --alphazero-require-simulator --alphazero-device cpu --battle-timeout 1800
time="2026-05-04T21:01:25-03:00" level=warning msg="Found orphan containers ([alphazero-train-offline-v2 alphazero-train-offline]) for this project. If you removed or renamed this service in your compose file, you can run this command with the --remove-orphans flag to clean it up."
 Container showdown-sim  Running
 Container showdown  Running
  VGC Bot 

: 

## 6. Cinco partidas contra cada rival

Usa el mismo checkpoint y la misma configuracion, pero con `--n 5`. Sirve para mostrar metricas menos ruidosas que una sola partida, sin esperar una evaluacion larga.

In [15]:
five_game_metrics = []
five_game_outputs = {}

for rival in RIVALS:
    print("\n" + "=" * 80)
    print(f"AlphaZero MCTS vs {rival} - 5 partidas")
    output_path = OUTPUT_DIR / f"five_vs_{rival}.txt"
    output = docker_compose(*play_command(rival, 5), save_to=output_path)
    five_game_outputs[rival] = output
    five_game_metrics.append(parse_play_output(output, rival, 5))

show_table(five_game_metrics)


AlphaZero MCTS vs random - 5 partidas
$ docker compose run --rm trainer python -u play.py --mode challenge --n 5 --p1 alphazero_mcts --p2 random --server showdown:8000 --format gen9vgc2026regi --team team.txt --alphazero-checkpoint checkpoints/alphazero_mcts_ppo_offline_d2/best.pt --alphazero-simulations 128 --alphazero-depth 2 --alphazero-max-candidates 96 --alphazero-simulator-max-choices 8 --alphazero-simulator-opponent-policy robust --alphazero-simulator-robust-worst-weight 0.35 --alphazero-simulator-timeout 180 --alphazero-live-state-url http://showdown:9002 --alphazero-require-simulator --alphazero-device cpu --battle-timeout 1800
time="2026-05-04T20:13:23-03:00" level=warning msg="Found orphan containers ([alphazero-train-offline-v2 alphazero-train-offline]) for this project. If you removed or renamed this service in your compose file, you can run this command with the --remove-orphans flag to clean it up."
 Container showdown-sim  Running
 Container showdown  Running
  VGC Bot

## 7. Ver detalles de partidas guardadas

In [9]:
def show_final_lines(path: Path) -> None:
    text = path.read_text(encoding="utf-8", errors="replace")
    lines = [line for line in text.splitlines() if "Final:" in line or line.strip().startswith("[")]
    print(f"Archivo: {path}")
    print("\n".join(lines[-30:]))

for path in sorted(OUTPUT_DIR.glob("*.txt")):
    if path.name.startswith(("single_vs_", "five_vs_")):
        show_final_lines(path)
        print("\n" + "-" * 80)

Archivo: notebooks\demo_outputs\five_vs_alphazero_mcts.txt
  Final: battle-gen9vgc2026regi-14 -> VICTORIA en 10 turnos.
  Final: battle-gen9vgc2026regi-15 -> DERROTA en 8 turnos.
  Final: battle-gen9vgc2026regi-16 -> VICTORIA en 8 turnos.
  Final: battle-gen9vgc2026regi-17 -> VICTORIA en 13 turnos.
  Final: battle-gen9vgc2026regi-18 -> DERROTA en 43 turnos.
    [VICTORIA] battle-gen9vgc2026regi-14 - 10 turnos
    [DERROTA] battle-gen9vgc2026regi-15 - 8 turnos
    [VICTORIA] battle-gen9vgc2026regi-16 - 8 turnos
    [VICTORIA] battle-gen9vgc2026regi-17 - 13 turnos
    [DERROTA] battle-gen9vgc2026regi-18 - 43 turnos

--------------------------------------------------------------------------------
Archivo: notebooks\demo_outputs\five_vs_greedy.txt
  Final: battle-gen9vgc2026regi-9 -> DERROTA en 7 turnos.
  Final: battle-gen9vgc2026regi-10 -> DERROTA en 11 turnos.
  Final: battle-gen9vgc2026regi-11 -> VICTORIA en 4 turnos.
  Final: battle-gen9vgc2026regi-12 -> VICTORIA en 8 turnos.
  Final:

## 8. Logs de entrenamiento recientes

In [10]:
def show_jsonl_tail(path: Path, n: int = 8) -> None:
    if not path.exists():
        print(f"No existe {path}")
        return
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()[-n:]
    for line in lines:
        try:
            print(json.dumps(json.loads(line), indent=2))
        except json.JSONDecodeError:
            print(line)

show_jsonl_tail(Path("logs/alphazero_train_offline_d2_streaming.jsonl"), n=8)

{
  "ts": "2026-04-27T20:28:30+0000",
  "event": "ppo_epoch",
  "memory": {
    "vmpeak": "4417764 kB",
    "vmsize": "4397060 kB",
    "vmhwm": "760760 kB",
    "vmrss": "739724 kB"
  },
  "iteration": 10,
  "epoch": 4,
  "samples": 20000,
  "elapsed_sec": 61.4,
  "loss": 1.728321,
  "ppo": 0.235554,
  "value": 0.000214,
  "mcts_ce": 0.997143,
  "entropy": 1.007318
}
{
  "ts": "2026-04-27T20:29:31+0000",
  "event": "ppo_epoch",
  "memory": {
    "vmpeak": "4417764 kB",
    "vmsize": "4399684 kB",
    "vmhwm": "760760 kB",
    "vmrss": "742540 kB"
  },
  "iteration": 10,
  "epoch": 5,
  "samples": 20000,
  "elapsed_sec": 60.929,
  "loss": 1.728196,
  "ppo": 0.235533,
  "value": 0.000219,
  "mcts_ce": 0.997072,
  "entropy": 1.0071
}
{
  "ts": "2026-04-27T20:30:32+0000",
  "event": "ppo_epoch",
  "memory": {
    "vmpeak": "4417764 kB",
    "vmsize": "4397060 kB",
    "vmhwm": "760760 kB",
    "vmrss": "740128 kB"
  },
  "iteration": 10,
  "epoch": 6,
  "samples": 20000,
  "elapsed_sec": 

## 9. Visualizaciones

In [16]:
import math
import sys
from collections import defaultdict
from html import escape

try:
    from IPython.display import HTML, display
    HAS_IPYTHON_HTML = True
except Exception:
    HTML = None
    HAS_IPYTHON_HTML = False


# Si un intento anterior dejo numpy parcialmente importado/roto, IPython puede fallar al formatear salidas.
_bad_numpy = sys.modules.get("numpy")
if _bad_numpy is not None and (not hasattr(_bad_numpy, "float64") or not hasattr(_bad_numpy, "__version__")):
    sys.modules.pop("numpy", None)

# En algunos kernels el PlainTextFormatter se inicializa tarde y vuelve a mirar sys.modules["numpy"].
# Forzamos una configuracion minima antes de imprimir/renderizar nada.
try:
    _ip = get_ipython()
except NameError:
    _ip = None
if _ip is not None:
    try:
        _plain_formatter = _ip.display_formatter.formatters.get("text/plain")
        if _plain_formatter is not None:
            _plain_formatter.type_printers = {}
    except Exception:
        pass

TRAIN_LOG_PATH = Path("logs/alphazero_train_offline_d2_streaming.jsonl")


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    return rows


def memory_kb_to_mb(value: str | int | float | None) -> float | None:
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value) / 1024.0
    match = re.search(r"(\d+(?:\.\d+)?)", str(value))
    return float(match.group(1)) / 1024.0 if match else None


def fmt(value, digits: int = 3) -> str:
    if value is None:
        return ""
    if isinstance(value, float):
        return f"{value:.{digits}f}"
    return str(value)


def display_html(html: str) -> None:
    if HAS_IPYTHON_HTML:
        try:
            # raw=True evita el PlainTextFormatter de IPython, que puede romperse si numpy esta mal instalado.
            display({"text/html": html}, raw=True)
            return
        except Exception as exc:
            print(f"No pude renderizar HTML en Jupyter: {type(exc).__name__}: {exc}")
    print(html)


def table(rows: list[dict], columns: list[str] | None = None):
    if not rows:
        print("Sin datos para mostrar.")
        return rows
    if columns is None:
        columns = sorted({key for row in rows for key in row})

    html = [
        "<div style='overflow-x:auto; margin: 8px 0 16px 0;'>",
        "<table style='border-collapse:collapse; font-family:system-ui, Segoe UI, sans-serif; font-size:13px;'>",
        "<thead><tr>",
    ]
    for col in columns:
        html.append(f"<th style='text-align:left; border-bottom:2px solid #bbb; padding:6px 10px;'>{escape(col)}</th>")
    html.append("</tr></thead><tbody>")
    for row in rows:
        html.append("<tr>")
        for col in columns:
            html.append(f"<td style='border-bottom:1px solid #ddd; padding:5px 10px; white-space:nowrap;'>{escape(fmt(row.get(col)))}</td>")
        html.append("</tr>")
    html.append("</tbody></table></div>")
    display_html("".join(html))
    return rows


def svg_card(title: str, body: str, width: int = 760, height: int = 320) -> str:
    return f"""
    <div style='margin:12px 0; max-width:{width}px; border:1px solid #ddd; border-radius:6px; padding:10px; background:#fff;'>
      <div style='font:600 15px system-ui, Segoe UI, sans-serif; margin-bottom:8px;'>{escape(title)}</div>
      <svg width='100%' viewBox='0 0 {width} {height}' role='img' aria-label='{escape(title)}'>
        {body}
      </svg>
    </div>
    """


def bar_chart(title: str, items: list[tuple[str, float]], *, max_value: float | None = None, unit: str = "", digits: int = 1, color: str = "#2f7d6d") -> str:
    clean = [(label, float(value or 0)) for label, value in items]
    if not clean:
        return ""
    width = 760
    row_h = 34
    top = 20
    left = 160
    right = 90
    chart_w = width - left - right
    height = top + row_h * len(clean) + 20
    scale_max = max_value or max(value for _, value in clean) or 1.0
    parts = []
    for i, (label, value) in enumerate(clean):
        y = top + i * row_h
        bar_w = max(1, chart_w * value / scale_max)
        parts.append(f"<text x='0' y='{y + 18}' font-size='12' font-family='system-ui, Segoe UI, sans-serif'>{escape(label)}</text>")
        parts.append(f"<rect x='{left}' y='{y + 5}' width='{bar_w:.2f}' height='18' rx='3' fill='{color}'></rect>")
        parts.append(f"<text x='{left + bar_w + 8:.2f}' y='{y + 19}' font-size='12' font-family='system-ui, Segoe UI, sans-serif'>{value:.{digits}f}{escape(unit)}</text>")
    return svg_card(title, "".join(parts), width=width, height=height)


def line_chart(title: str, series: list[tuple[str, list[tuple[float, float | None]], str]], *, x_label: str = "", y_label: str = "") -> str:
    width = 760
    height = 300
    left = 56
    right = 20
    top = 22
    bottom = 44
    plot_w = width - left - right
    plot_h = height - top - bottom

    all_points = [(x, y) for _, points, _ in series for x, y in points if y is not None]
    if not all_points:
        return ""
    xs = [x for x, _ in all_points]
    ys = [y for _, y in all_points]
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    if xmin == xmax:
        xmin -= 1
        xmax += 1
    if ymin == ymax:
        ymin -= 1
        ymax += 1
    ypad = (ymax - ymin) * 0.08
    ymin -= ypad
    ymax += ypad

    def sx(x):
        return left + (x - xmin) / (xmax - xmin) * plot_w

    def sy(y):
        return top + (ymax - y) / (ymax - ymin) * plot_h

    parts = [
        f"<line x1='{left}' y1='{top}' x2='{left}' y2='{top + plot_h}' stroke='#999' />",
        f"<line x1='{left}' y1='{top + plot_h}' x2='{left + plot_w}' y2='{top + plot_h}' stroke='#999' />",
        f"<text x='{left}' y='{height - 10}' font-size='11' font-family='system-ui, Segoe UI, sans-serif'>{escape(x_label)}</text>",
        f"<text x='0' y='14' font-size='11' font-family='system-ui, Segoe UI, sans-serif'>{escape(y_label)}</text>",
        f"<text x='{left}' y='{top + plot_h + 18}' font-size='10' font-family='system-ui, Segoe UI, sans-serif'>{xmin:.0f}</text>",
        f"<text x='{left + plot_w - 22}' y='{top + plot_h + 18}' font-size='10' font-family='system-ui, Segoe UI, sans-serif'>{xmax:.0f}</text>",
        f"<text x='6' y='{sy(ymax) + 4:.1f}' font-size='10' font-family='system-ui, Segoe UI, sans-serif'>{ymax:.3g}</text>",
        f"<text x='6' y='{sy(ymin) + 4:.1f}' font-size='10' font-family='system-ui, Segoe UI, sans-serif'>{ymin:.3g}</text>",
    ]

    legend_x = left + 8
    for idx, (name, points, color) in enumerate(series):
        clean = [(x, y) for x, y in points if y is not None]
        if not clean:
            continue
        path = " ".join(("M" if i == 0 else "L") + f" {sx(x):.2f} {sy(y):.2f}" for i, (x, y) in enumerate(clean))
        parts.append(f"<path d='{path}' fill='none' stroke='{color}' stroke-width='2.2' />")
        for x, y in clean:
            parts.append(f"<circle cx='{sx(x):.2f}' cy='{sy(y):.2f}' r='2.6' fill='{color}' />")
        ly = top + 14 + idx * 18
        parts.append(f"<rect x='{legend_x}' y='{ly - 9}' width='10' height='10' fill='{color}' />")
        parts.append(f"<text x='{legend_x + 15}' y='{ly}' font-size='11' font-family='system-ui, Segoe UI, sans-serif'>{escape(name)}</text>")
    return svg_card(title, "".join(parts), width=width, height=height)


def collect_play_metrics(output_dir: Path = OUTPUT_DIR) -> list[dict]:
    rows = []
    for path in sorted(output_dir.glob("*_vs_*.txt")):
        match = re.match(r"(?P<kind>single|five)_vs_(?P<rival>.+)\.txt", path.name)
        if not match:
            continue
        kind = match.group("kind")
        rival = match.group("rival")
        n = 1 if kind == "single" else 5
        text = path.read_text(encoding="utf-8", errors="replace")
        parsed = parse_play_output(text, rival, n)
        parsed["run"] = kind
        parsed["file"] = str(path)
        rows.append(parsed)
    return rows


def collect_training_metrics(log_path: Path = TRAIN_LOG_PATH) -> tuple[list[dict], list[dict]]:
    events = read_jsonl(log_path)
    starts = {row.get("iteration"): row for row in events if row.get("event") == "iteration_start"}
    rollouts = {row.get("iteration"): row for row in events if row.get("event") == "rollout_collected"}
    ppo_starts = {row.get("iteration"): row for row in events if row.get("event") == "ppo_start"}
    epochs = [row for row in events if row.get("event") == "ppo_epoch"]

    epoch_rows = []
    for idx, row in enumerate(epochs, start=1):
        mem = row.get("memory", {}) or {}
        epoch_rows.append({
            "step": idx,
            "iteration": row.get("iteration"),
            "opponent": starts.get(row.get("iteration"), {}).get("opponent"),
            "epoch": row.get("epoch"),
            "loss": row.get("loss"),
            "ppo": row.get("ppo"),
            "value": row.get("value"),
            "mcts_ce": row.get("mcts_ce"),
            "entropy": row.get("entropy"),
            "elapsed_sec": row.get("elapsed_sec"),
            "vmrss_mb": memory_kb_to_mb(mem.get("vmrss")),
        })

    by_iteration = defaultdict(list)
    for row in epoch_rows:
        by_iteration[row["iteration"]].append(row)

    iteration_rows = []
    for iteration in sorted(by_iteration):
        rows = by_iteration[iteration]
        final = max(rows, key=lambda row: row.get("epoch") or 0)
        rollout = rollouts.get(iteration, {})
        ppo_start = ppo_starts.get(iteration, {})
        iteration_rows.append({
            "iteration": iteration,
            "opponent": starts.get(iteration, {}).get("opponent"),
            "decisions": rollout.get("decisions"),
            "samples": ppo_start.get("samples"),
            "forced": ppo_start.get("forced"),
            "real_sim": ppo_start.get("real_sim"),
            "fallback_sim": ppo_start.get("fallback_sim"),
            "invalid_rows": ppo_start.get("invalid_rows"),
            "final_loss": final.get("loss"),
            "final_mcts_ce": final.get("mcts_ce"),
            "final_entropy": final.get("entropy"),
            "avg_epoch_sec": sum((row.get("elapsed_sec") or 0) for row in rows) / max(len(rows), 1),
            "vmrss_mb": final.get("vmrss_mb"),
        })
    return epoch_rows, iteration_rows


play_metrics = collect_play_metrics()
epoch_metrics, iteration_metrics = collect_training_metrics()

print(f"Resultados de partidas encontrados: {len(play_metrics)} archivo(s)")
print(f"Epochs de entrenamiento encontrados: {len(epoch_metrics)}")
print(f"Iteraciones con entrenamiento encontradas: {len(iteration_metrics)}")


Resultados de partidas encontrados: 6 archivo(s)
Epochs de entrenamiento encontrados: 80
Iteraciones con entrenamiento encontradas: 10


### Resultados de partidas

In [17]:
play_df = table(
    play_metrics,
    ["run", "rival", "n", "wins", "losses", "win_rate", "avg_turns", "min_turns", "max_turns", "file"],
)

if play_metrics:
    charts = []
    charts.append(bar_chart(
        "Win rate del modelo",
        [(f"{row['rival']} / {row['run']}", 100 * (row.get("win_rate") or 0)) for row in play_metrics],
        max_value=100,
        unit="%",
        digits=0,
        color="#2f7d6d",
    ))
    charts.append(bar_chart(
        "Duracion promedio",
        [(f"{row['rival']} / {row['run']}", row.get("avg_turns") or 0) for row in play_metrics],
        unit=" turnos",
        digits=1,
        color="#b45f4d",
    ))
    display_html("".join(charts))
else:
    print("Todavia no hay salidas de partidas guardadas para graficar.")


run,rival,n,wins,losses,win_rate,avg_turns,min_turns,max_turns,file
five,alphazero_mcts,5,3,2,0.600,16.400,8,43,notebooks\demo_outputs\five_vs_alphazero_mcts.txt
five,greedy,5,2,3,0.400,8.800,4,14,notebooks\demo_outputs\five_vs_greedy.txt
five,random,5,5,0,1.000,12.000,6,19,notebooks\demo_outputs\five_vs_random.txt
single,alphazero_mcts,1,0,1,0.000,13.000,13,13,notebooks\demo_outputs\single_vs_alphazero_mcts.txt
single,greedy,1,0,1,0.000,7.000,7,7,notebooks\demo_outputs\single_vs_greedy.txt
single,random,1,1,0,1.000,17.000,17,17,notebooks\demo_outputs\single_vs_random.txt


Win rate del modelo 
 
 alphazero_mcts / five 60% greedy / five 40% random / five 100% alphazero_mcts / single 0% greedy / single 0% random / single 100% 
 
 
 
 
 Duracion promedio 
 
 alphazero_mcts / five 16.4 turnos greedy / five 8.8 turnos random / five 12.0 turnos alphazero_mcts / single 13.0 turnos greedy / single 7.0 turnos random / single 17.0 turnos

### Curvas de entrenamiento

In [18]:
train_df = table(
    iteration_metrics,
    ["iteration", "opponent", "decisions", "samples", "forced", "real_sim", "fallback_sim", "invalid_rows", "final_loss", "final_mcts_ce", "final_entropy", "avg_epoch_sec", "vmrss_mb"],
)

if epoch_metrics:
    series_specs = [
        ("loss", "Loss total", "#2f7d6d"),
        ("mcts_ce", "CE contra MCTS", "#4f6f9f"),
        ("ppo", "PPO", "#b45f4d"),
        ("entropy", "Entropia", "#7a5aa6"),
    ]
    charts = []
    for key, title, color in series_specs:
        charts.append(line_chart(
            title,
            [(key, [(row["step"], row.get(key)) for row in epoch_metrics], color)],
            x_label="Epoch global",
            y_label=key,
        ))
    display_html("".join(charts))
else:
    print("No encontre eventos ppo_epoch en el log de entrenamiento.")


iteration,opponent,decisions,samples,forced,real_sim,fallback_sim,invalid_rows,final_loss,final_mcts_ce,final_entropy,avg_epoch_sec,vmrss_mb
1,random,849,20000,101,20000,0,0,1.690,0.965,0.982,64.673,681.703
2,greedy,846,20000,98,20000,0,0,1.689,0.963,0.980,65.969,700.492
3,self,1007,20000,99,20000,0,0,1.685,0.962,0.979,64.934,696.547
4,greedy,1182,20000,101,20000,0,0,1.695,0.970,0.986,65.731,695.582
5,self,1141,20000,201,20000,0,0,1.698,0.975,0.990,66.967,694.496
6,random,1070,20000,191,20000,0,0,1.690,0.965,0.980,64.431,694.500
7,greedy,651,20000,192,20000,0,0,1.663,0.946,0.959,66.912,694.527
8,self,1199,20000,201,20000,0,0,1.687,0.968,0.980,59.145,705.941
9,random,657,20000,198,20000,0,0,1.706,0.983,0.994,62.875,694.117
10,greedy,872,20000,197,20000,0,0,1.728,0.997,1.007,60.937,725.301


Loss total 
 
 Epoch global loss 1 80 1.73 1.66 <path d='M 56.00 148.26 L 64.66 152.68 L 73.32 152.42 L 81.97 152.21 L 90.63 153.99 L 99.29 154.47 L 107.95 153.98 L 116.61 154.40 L 125.27 155.00 L 133.92 157.77 L 142.58 157.54 L 151.24 157.32 L 159.90 158.92 L 168.56 159.19 L 177.22 159.33 L 185.87 159.33 L 194.53 166.73 L 203.19 167.71 L 211.85 168.62 L 220.51 170.07 L 229.16 171.06 L 237.82 170.41 L 246.48 171.35 L 255.14 171.07 L 263.80 134.90 L 272.46 136.55 L 281.11 137.00 L 289.77 139.13 L 298.43 137.43 L 307.09 138.81 L 315.75 138.94 L 324.41 139.46 L 333.06 121.62 L 341.72 127.38 L 350.38 128.57 L 359.04 130.39 L 367.70 131.59 L 376.35 130.43 L 385.01 130.48 L 393.67 132.44 L 402.33 150.60 L 410.99 152.94 L 419.65 153.71 L 428.30 152.48 L 436.96 154.48 L 445.62 153.59 L 454.28 153.73 L 462.94 155.11 L 471.59 237.32 L 480.25 237.88 L 488.91 238.75 L 497.57 238.10 L 506.23 237.75 L 514.89 239.37 L 523.54 239.86 L 532.20 238.41 L 540.86 156.15 L 549.52 159.38 L 558.18 162.60 L 566.84 163.28 L 575.49 164.49 L 584.15 163.43 L 592.81 162.75 L 601.47 164.86 L 610.13 100.91 L 618.78 104.01 L 627.44 104.79 L 636.10 104.94 L 644.76 105.12 L 653.42 104.80 L 662.08 106.16 L 670.73 106.07 L 679.39 38.14 L 688.05 39.46 L 696.71 39.73 L 705.37 39.32 L 714.03 39.70 L 722.68 40.44 L 731.34 40.33 L 740.00 40.95' fill='none' stroke='#2f7d6d' stroke-width='2.2' /> loss 
 
 
 
 
 CE contra MCTS 
 
 Epoch global mcts_ce 1 80 1 0.942 <path d='M 56.00 160.36 L 64.66 163.17 L 73.32 162.67 L 81.97 162.42 L 90.63 163.88 L 99.29 163.66 L 107.95 163.61 L 116.61 163.83 L 125.27 169.34 L 133.92 171.28 L 142.58 170.85 L 151.24 170.69 L 159.90 172.04 L 168.56 171.71 L 177.22 172.20 L 185.87 172.04 L 194.53 175.38 L 203.19 175.70 L 211.85 176.48 L 220.51 177.69 L 229.16 178.30 L 237.82 177.94 L 246.48 178.65 L 255.14 178.23 L 263.80 140.92 L 272.46 141.53 L 281.11 141.71 L 289.77 143.49 L 298.43 141.79 L 307.09 142.96 L 315.75 143.04 L 324.41 143.68 L 333.06 118.62 L 341.72 123.04 L 350.38 124.31 L 359.04 125.38 L 367.70 126.50 L 376.35 125.42 L 385.01 125.74 L 393.67 127.11 L 402.33 163.98 L 410.99 165.13 L 419.65 165.39 L 428.30 164.63 L 436.96 166.52 L 445.62 165.15 L 454.28 165.59 L 462.94 166.44 L 471.59 238.43 L 480.25 238.62 L 488.91 239.15 L 497.57 238.61 L 506.23 238.25 L 514.89 239.55 L 523.54 239.86 L 532.20 238.80 L 540.86 148.54 L 549.52 150.44 L 558.18 153.14 L 566.84 153.39 L 575.49 154.45 L 584.15 153.46 L 592.81 152.79 L 601.47 154.48 L 610.13 90.71 L 618.78 92.21 L 627.44 92.82 L 636.10 93.20 L 644.76 92.84 L 653.42 92.84 L 662.08 93.66 L 670.73 93.59 L 679.39 38.36 L 688.05 38.53 L 696.71 38.85 L 705.37 38.14 L 714.03 38.42 L 722.68 39.16 L 731.34 38.91 L 740.00 39.25' fill='none' stroke='#4f6f9f' stroke-width='2.2' /> mcts_ce 
 
 
 
 
 PPO 
 
 Epoch global ppo 1 80 0.248 0.233 <path d='M 56.00 59.94 L 64.66 64.44 L 73.32 65.45 L 81.97 65.47 L 90.63 65.73 L 99.29 69.42 L 107.95 67.01 L 116.61 67.81 L 125.27 38.14 L 133.92 40.87 L 142.58 41.96 L 151.24 41.52 L 159.90 41.70 L 168.56 45.13 L 177.22 42.64 L 185.87 43.52 L 194.53 63.17 L 203.19 66.25 L 211.85 66.27 L 220.51 66.41 L 229.16 67.94 L 237.82 66.48 L 246.48 67.18 L 255.14 68.25 L 263.80 103.57 L 272.46 108.59 L 281.11 109.72 L 289.77 110.04 L 298.43 111.54 L 307.09 111.69 L 315.75 111.78 L 324.41 110.42 L 333.06 170.07 L 341.72 172.89 L 350.38 171.44 L 359.04 174.48 L 367.70 174.07 L 376.35 174.38 L 385.01 172.59 L 393.67 174.61 L 402.33 45.52 L 410.99 50.92 L 419.65 53.51 L 428.30 51.49 L 436.96 50.55 L 445.62 54.15 L 454.28 52.23 L 462.94 54.34 L 471.59 53.46 L 480.25 55.22 L 488.91 56.55 L 497.57 56.49 L 506.23 56.75 L 514.89 57.35 L 523.54 58.05 L 532.20 56.78 L 540.86 174.16 L 549.52 178.48 L 558.18 178.40 L 566.84 179.98 L 575.49 179.51 L 584.15 179.69 L 592.81 180.03 L 601.47 180.84 L 610.13 232.67 L 618.78 238.32 L 627.44 238.33 L 636.10 236.61 L 644.76 239.52 L 653.42 237.80 L 662.08 239.86 L 670.73 239.59 L 679.39 214.61 L 688.05 220.47 L 696.71 220.06 L 705.37 221.98

### Progreso por iteracion

In [19]:
if iteration_metrics:
    charts = []
    charts.append(bar_chart(
        "Decisiones recolectadas por iteracion",
        [(f"{row['iteration']} ({row.get('opponent') or '?'})", row.get("decisions") or 0) for row in iteration_metrics],
        unit="",
        digits=0,
        color="#4f6f9f",
    ))
    charts.append(line_chart(
        "Loss final por iteracion",
        [("final_loss", [(row["iteration"], row.get("final_loss")) for row in iteration_metrics], "#2f7d6d")],
        x_label="Iteracion",
        y_label="loss",
    ))
    charts.append(bar_chart(
        "Tiempo promedio por epoch",
        [(str(row["iteration"]), row.get("avg_epoch_sec") or 0) for row in iteration_metrics],
        unit="s",
        digits=1,
        color="#b45f4d",
    ))
    charts.append(line_chart(
        "Memoria RSS al final",
        [("vmrss_mb", [(row["iteration"], row.get("vmrss_mb")) for row in iteration_metrics], "#7a5aa6")],
        x_label="Iteracion",
        y_label="MB",
    ))
    display_html("".join(charts))
else:
    print("No hay iteraciones completas para graficar.")


Decisiones recolectadas por iteracion 
 
 1 (random) 849 2 (greedy) 846 3 (self) 1007 4 (greedy) 1182 5 (self) 1141 6 (random) 1070 7 (greedy) 651 8 (self) 1199 9 (random) 657 10 (greedy) 872 
 
 
 
 
 Loss final por iteracion 
 
 Iteracion loss 1 10 1.73 1.66 final_loss 
 
 
 
 
 Tiempo promedio por epoch 
 
 1 64.7s 2 66.0s 3 64.9s 4 65.7s 5 67.0s 6 64.4s 7 66.9s 8 59.1s 9 62.9s 10 60.9s 
 
 
 
 
 Memoria RSS al final 
 
 Iteracion MB 1 10 729 678 vmrss_mb

### Comparacion dentro de una iteracion

In [20]:
ITERATION_TO_INSPECT = max([row["iteration"] for row in epoch_metrics], default=None)

if ITERATION_TO_INSPECT is None:
    print("No hay epochs para inspeccionar.")
else:
    selected = [row for row in epoch_metrics if row["iteration"] == ITERATION_TO_INSPECT]
    print(f"Inspeccionando iteracion {ITERATION_TO_INSPECT}")
    table(selected, ["iteration", "opponent", "epoch", "loss", "ppo", "value", "mcts_ce", "entropy", "elapsed_sec", "vmrss_mb"])

    display_html(line_chart(
        f"Metricas por epoch - iteracion {ITERATION_TO_INSPECT}",
        [
            ("loss", [(row["epoch"], row.get("loss")) for row in selected], "#2f7d6d"),
            ("mcts_ce", [(row["epoch"], row.get("mcts_ce")) for row in selected], "#4f6f9f"),
            ("entropy", [(row["epoch"], row.get("entropy")) for row in selected], "#7a5aa6"),
        ],
        x_label="Epoch",
        y_label="valor",
    ))


Inspeccionando iteracion 10


iteration,opponent,epoch,loss,ppo,value,mcts_ce,entropy,elapsed_sec,vmrss_mb
10,greedy,1,1.729,0.236,0.000,0.997,1.007,60.017,723.367
10,greedy,2,1.728,0.236,0.000,0.997,1.007,60.729,722.707
10,greedy,3,1.728,0.236,0.000,0.997,1.007,60.470,722.668
10,greedy,4,1.728,0.236,0.000,0.997,1.007,61.400,722.387
10,greedy,5,1.728,0.236,0.000,0.997,1.007,60.929,725.137
10,greedy,6,1.728,0.236,0.000,0.997,1.007,61.056,722.781
10,greedy,7,1.728,0.236,0.000,0.997,1.007,61.114,722.523
10,greedy,8,1.728,0.235,0.000,0.997,1.007,61.780,725.301


Metricas por epoch - iteracion 10 
 
 Epoch valor 1 8 1.79 0.938 loss mcts_ce entropy